In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%pip install highway-env
%pip install stable-baselines3

In [ ]:
%cd /content/drive/MyDrive/cs-272-final-project/custom/baseline/ppo_204800

/content/drive/MyDrive/cs-272-final-project/custom/baseline/ppo_204800


In [ ]:
# Ignore warnings
import warnings
warnings.filterwarnings("ignore",category=DeprecationWarning,)
warnings.filterwarnings("ignore", message=".*Kernel._parent_header.*")

In [ ]:
# Learning curve (episodes on X) from Monitor CSVs
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def plot_learning_curve_monitor(monitor_dir: str, out_png: str, smooth_window: int = 50):
    files = sorted(glob.glob(os.path.join(monitor_dir, "**", "*.csv"), recursive=True))
    if not files:
        raise FileNotFoundError(f"No monitor CSVs found under: {monitor_dir}")

    dfs = []
    for f in files:
        try:
            df = pd.read_csv(f, comment="#")
            if "r" in df.columns:
                dfs.append(df[["r"]].copy())
        except Exception:
            pass

    if not dfs:
        raise RuntimeError("Found CSVs but no episodic returns (column 'r').")

    log = pd.concat(dfs, ignore_index=True)
    log["episode"] = np.arange(1, len(log) + 1)

    win = max(1, min(smooth_window, len(log)))
    log["r_smooth"] = log["r"].rolling(win, min_periods=1).mean()

    os.makedirs(os.path.dirname(out_png) or ".", exist_ok=True)
    plt.figure()
    plt.plot(log["episode"], log["r_smooth"])
    plt.xlabel("Training Episodes")
    plt.ylabel("Mean Episodic Return (smoothed)")
    plt.title("Learning Curve — Custom Env PPO")
    plt.tight_layout()
    plt.savefig(out_png, dpi=140)
    plt.close()
    print(f"Saved: {out_png}")


In [ ]:
# 500-episode deterministic evaluation violin plot
import os
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
from stable_baselines3 import PPO

def evaluate_and_plot_violin_custom(model_path: str, out_png: str, n_episodes: int = 500, seed: int = 0):
    env = gym.make("MyCustomEnv-v0")
    model = PPO.load(model_path)

    returns = []
    for ep in range(n_episodes):
        obs, info = env.reset(seed=seed)
        done = False
        ep_ret = 0.0
        while not done:
            action, _ = model.predict(obs, deterministic=True)  # no exploration
            obs, reward, terminated, truncated, info = env.step(action)
            ep_ret += reward
            done = terminated or truncated
        returns.append(ep_ret)

    env.close()

    os.makedirs(os.path.dirname(out_png) or ".", exist_ok=True)
    plt.figure()
    plt.violinplot([returns], showmeans=True)
    plt.xticks([1], ["PPO"])
    plt.ylabel("Episodic Return (n=500, deterministic)")
    plt.title("Performance Test — Custom Env PPO")
    plt.tight_layout()
    plt.savefig(out_png, dpi=140)
    plt.close()
    print(f"Saved: {out_png}")



In [ ]:
import os
import gymnasium as gym
from custom_env import AccidentEnv
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env

# Optional: register (you can also skip this and use AccidentEnv() directly)
try:
    gym.register(id="MyCustomEnv-v0", entry_point="custom_env:AccidentEnv")
except gym.error.RegistrationError:
    print("Environment already registered.")

def main():
    BASE_DIR = os.getcwd()
    MODEL_DIR = os.path.join(BASE_DIR, "models")
    PLOT_DIR = os.path.join(BASE_DIR, "plots")
    TBOARD_DIR = os.path.join(BASE_DIR, "runs_tboard")
    os.makedirs(TBOARD_DIR, exist_ok=True)
    os.makedirs(PLOT_DIR, exist_ok=True)
    os.makedirs(MODEL_DIR, exist_ok=True)

    # === Log training episodes via Monitor CSVs ===
    MONITOR_DIR = os.path.join(BASE_DIR, "runs_monitor")
    os.makedirs(MONITOR_DIR, exist_ok=True)

    # Vectorized training env with monitor logging
    vec_env = make_vec_env(
        "MyCustomEnv-v0",
        n_envs=4,
        seed=42,
        monitor_dir=MONITOR_DIR,   # <-- this logs training episodes
    )

    # PPO model
    model = PPO(
        "MlpPolicy",
        vec_env,
        verbose=1,
        n_steps=2048,
        batch_size=64,
        learning_rate=3e-4,
        ent_coef=0.0,
        tensorboard_log=TBOARD_DIR,
    )

    # Train
    total_timesteps = int(204800)
    model.learn(total_timesteps=total_timesteps, progress_bar=False, tb_log_name="PPO_204800_steps")


    # Save
    save_path = os.path.join(MODEL_DIR, "ppo_204800_steps_custom_env")
    model.save(save_path)
    print(f"Model saved to: {save_path}.zip")

    plot_learning_curve_monitor(MONITOR_DIR, os.path.join(PLOT_DIR, "ppo_204800_steps_learning.png"))
    evaluate_and_plot_violin_custom(save_path, os.path.join(PLOT_DIR, "ppo_204800_steps_violin.png"))

    # Clean up
    vec_env.close()

if __name__ == "__main__":
    main()

Using cpu device
Logging to /content/drive/MyDrive/cs-272-final-project/custom/baseline/ppo_204800/runs_tboard/PPO_204800_steps_1


/usr/local/lib/python3.12/dist-packages/gymnasium/envs/registration.py:636: UserWarning: WARN: Overriding environment MyCustomEnv-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 8.96     |
|    ep_rew_mean     | 7.11     |
| time/              |          |
|    fps             | 15       |
|    iterations      | 1        |
|    time_elapsed    | 543      |
|    total_timesteps | 8192     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 10.8        |
|    ep_rew_mean          | 8.73        |
| time/                   |             |
|    fps                  | 14          |
|    iterations           | 2           |
|    time_elapsed         | 1101        |
|    total_timesteps      | 16384       |
| train/                  |             |
|    approx_kl            | 0.015514351 |
|    clip_fraction        | 0.221       |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.6        |
|    explained_variance   | -0.0115     |
|    learning_rate        | 0.